In [ ]:
import os
from pathlib import Path
from data_preparation.sources.hub.catalog_library import CatalogSearchUtils, get_auth_session
from shared.io import read_data, write_data, file_exists

In [ ]:
catalogue_endpoint = os.environ["CATALOG_URL"]
client_id = os.getenv("CATALOG_CLIENT_ID", "internal")

In [ ]:
WORK_PATH = Path("C:/work_sat")
RUN_IDENTIFIER = "central_greece"

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "central_greece_prediction.parquet"
gdf_parcels = read_data(str(PARCELS_PATH))
gdf_parcels.head()

In [ ]:
gdf_parcels = gdf_parcels.to_crs("EPSG:4326")
gdf_parcel = gdf_parcels[gdf_parcels["parcel_id"].astype(str) == "284266200"].copy()
gdf_parcel

In [ ]:
# convert geometry in the form of GeoJSON
geometry = gdf_parcel.iloc[0].geometry
geometry_geojson = geometry.__geo_interface__
geometry_geojson

In [ ]:
import shutil
from typing import Any, Callable, Iterable, Optional
from urllib.parse import unquote, urlparse
from pathlib import Path
import requests
import boto3
from botocore.client import BaseClient
from botocore.config import Config

from ml_classification.shared import logging

timeout: int = 60  # Replace with the desired timeout in seconds
output_root = Path("C:/work_sat/test")  # Replace with the actual output directory

def _filename_from_reference(reference: Any) -> str | None:
    """Extract a filename from a relative path or URL-like reference."""
    if not isinstance(reference, str) or not reference.strip():
        return None
    parsed = urlparse(reference.strip())
    path_value = parsed.path if parsed.scheme or parsed.netloc or parsed.query or parsed.fragment else reference.strip()
    filename = Path(unquote(path_value).replace("\\", "/")).name
    return filename if filename not in {"", "."} else None

def get_asset_filename(asset_key: str, asset_info: dict[str, Any]) -> str:
    """Return a readable filename from relative path, href, or href fallbacks."""
    filename = _filename_from_reference(asset_info.get("hub:relativePath"))
    if filename:
        return filename

    for href in _candidate_hrefs(asset_info):
        filename = _filename_from_reference(href)
        if filename:
            return filename
    return asset_key

def _template_base_hrefs(asset_info: dict[str, Any]) -> Iterable[str]:
    """Yield base URLs derived from STAC tile and point-query templates."""
    for template_key, template_info in asset_info.items():
        if not template_key.endswith(("-tiles-template", "-query-template")) or not isinstance(template_info, dict):
            continue
        template_href = template_info.get("href")
        if not isinstance(template_href, str) or not template_href:
            continue
        for marker in ("/tiles/", "/point/"):
            if marker in template_href:
                yield template_href.split(marker, maxsplit=1)[0]
                break

def _candidate_hrefs(asset_info: dict[str, Any]) -> list[str]:
    """Collect unique direct, alternate, and templated base asset URLs."""
    candidates: list[str] = []

    def add_candidate(value: Any) -> None:
        """Append one non-empty URL while preserving provider ordering."""
        if isinstance(value, str) and value.strip():
            candidates.append(value.strip())

    add_candidate(asset_info.get("href"))

    # STAC's alternate extension can expose authenticated or cloud-native copies.
    alternate_hrefs = asset_info.get("alternate")
    if isinstance(alternate_hrefs, dict):
        for alternate_info in alternate_hrefs.values():
            if isinstance(alternate_info, dict):
                add_candidate(alternate_info.get("href"))

    for template_href in _template_base_hrefs(asset_info):
        add_candidate(template_href)

    return list(dict.fromkeys(candidates))

def get_asset_download_href(asset_info: dict[str, Any]) -> str | None:
    """Return the preferred direct, alternate, or template-derived asset href."""
    return next(iter(_candidate_hrefs(asset_info)), None)

def _download_http_asset(
    download_href: str,
    destination_path: Path,
    session: requests.Session,
    timeout: float,
) -> None:
    """Stream one HTTP asset to disk and close its response."""
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    response = None
    try:
        response = session.get(
            download_href,
            stream=True,
            allow_redirects=True,
            timeout=timeout,
        )
        response.raise_for_status()
        with destination_path.open("wb") as output_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output_file.write(chunk)
    finally:
        if response is not None:
            response.close()

def _safe_int_env(env_name: str, default_value: int, min_value: int = 1) -> int:
    """Read an integer environment variable with safe fallback.

    Parameters
    ----------
    env_name : str
        Environment variable name.
    default_value : int
        Fallback value when variable is missing or invalid.
    min_value : int, optional
        Minimum accepted value.

    Returns
    -------
    int
        Parsed integer, clamped to min_value.

    """
    raw_value = os.getenv(env_name)
    if raw_value is None:
        return default_value
    try:
        return max(min_value, int(raw_value))
    except (TypeError, ValueError):
        return default_value

def _safe_bool_env(env_name: str, default_value: bool) -> bool:
    """Read a boolean environment variable with safe fallback."""
    raw_value = os.getenv(env_name)
    if raw_value is None:
        return default_value
    return raw_value.strip().lower() in {"1", "true", "yes", "y", "on"}

def _safe_float_env(env_name: str, default_value: float, min_value: float = 0.0) -> float:
    """Read a float environment variable with safe fallback.

    Parameters
    ----------
    env_name : str
        Environment variable name.
    default_value : float
        Fallback value when variable is missing or invalid.
    min_value : float, optional
        Minimum accepted value.

    Returns
    -------
    float
        Parsed float, clamped to min_value.

    """
    raw_value = os.getenv(env_name)
    if raw_value is None:
        return default_value
    try:
        return max(min_value, float(raw_value))
    except (TypeError, ValueError):
        return default_value

def get_boto3_client(
    *,
    key: str | None = None,
    secret: str | None = None,
    region: str | None = None,
    endpoint_url: str | None = None,
    session_token: str | None = None,
    addressing_style: str | None = None,  # "virtual" or "path"
    verify: bool | str | None = None,  # True/False or path to CA bundle
) -> BaseClient:
    """Create and return a boto3 S3 client.

    This function initializes a boto3 S3 client using provided parameters or environment variables.
    It supports both AWS S3 and S3-compatible endpoints by allowing custom endpoint URLs and
    addressing styles. The client is configured with reasonable defaults for timeouts and retries.

    Returns
    -------
    boto3.client
        A boto3 S3 client instance configured with the provided credentials.

    """
    key = os.getenv("AWS_ACCESS_KEY_ID") if key is None else key
    secret = os.getenv("AWS_SECRET_ACCESS_KEY") if secret is None else secret
    session_token = os.getenv("AWS_SESSION_TOKEN") if session_token is None else session_token
    endpoint_url = os.getenv("S3_ENDPOINT_URL") if endpoint_url is None else endpoint_url
    region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") if region is None else region
    max_pool_connections = _safe_int_env("S3_MAX_POOL_CONNECTIONS", default_value=32, min_value=1)
    tcp_keepalive = _safe_bool_env("S3_TCP_KEEPALIVE", default_value=True)

    # Safer default for S3-compatible endpoints.
    if addressing_style is None:
        addressing_style = "path" if endpoint_url else "virtual"

    connect_timeout = _safe_int_env("S3_CONNECT_TIMEOUT", default_value=60, min_value=1)
    read_timeout = _safe_int_env("S3_READ_TIMEOUT", default_value=300, min_value=1)
    retry_attempts = _safe_int_env("S3_MAX_RETRY_ATTEMPTS", default_value=10, min_value=1)
    retry_mode = os.getenv("S3_RETRY_MODE", "adaptive")

    cfg = Config(
        signature_version="s3v4",
        s3={"addressing_style": addressing_style},
        connect_timeout=connect_timeout,
        read_timeout=read_timeout,
        retries={"max_attempts": retry_attempts, "mode": retry_mode},
        max_pool_connections=max_pool_connections,
        tcp_keepalive=tcp_keepalive,
    )

    return boto3.client(
        "s3",
        aws_access_key_id=key,
        aws_secret_access_key=secret,
        aws_session_token=session_token,
        region_name=region,
        endpoint_url=endpoint_url,
        verify=True if verify is None else verify,
        config=cfg,
    )

def download_s3_file(
    s3_client: BaseClient,
    bucket_name: str,
    s3_key: str,
    local_file_path: str,
    key: str | None = None,
    secret: str | None = None,
    region: str | None = None,
) -> None:
    """Download a single object from Amazon S3 to a local filesystem path.

    Creates parent directories for local_file_path if they do not exist and then
    downloads the object identified by s3://{bucket_name}/{s3_key} to the given
    local destination. A message is logged on success.

    Parameters
    ----------
    s3_client : botocore.client.BaseClient
        An S3 client instance. (Note: this function currently overrides the provided
        client by calling get_boto3_client() internally.)
    bucket_name : str
        Name of the S3 bucket containing the object.
    s3_key : str
        Object key (path within the bucket) to download.
    local_file_path : str
        Absolute or relative path where the file will be saved locally.
    logger : logging.Logger, optional
        Logger instance for emitting an informational message after download.

    Returns
    -------
    None

    Raises
    ------
    botocore.exceptions.ClientError
        If the S3 download fails (e.g., object not found, access denied).
    OSError
        If local directories cannot be created or the file cannot be written.

    Notes
    -----
    - This function ensures that os.path.dirname(local_file_path) exists
    by calling os.makedirs(..., exist_ok=True).

    """
    # Always use a fresh client to avoid stale credentials/session state.
    # Keep s3_client argument for backward compatibility with existing callers.
    if not s3_client:
        s3_client = get_boto3_client(key=key, secret=secret, region=region)
    parent_dir = os.path.dirname(local_file_path)
    if parent_dir:
        os.makedirs(parent_dir, exist_ok=True)
    s3_client.download_file(bucket_name, s3_key, local_file_path)

def _download_s3_asset(
    download_href: str,
    destination_path: Path,
    s3_client: Any,
    s3_downloader: Callable[..., None],
) -> None:
    """Download one valid S3 URI to the requested output path."""
    parsed = urlparse(download_href)
    bucket_name = parsed.netloc
    s3_key = parsed.path.lstrip("/")
    if not bucket_name or not s3_key:
        raise ValueError(f"Invalid S3 href: {download_href}")

    s3_downloader(
        s3_client=s3_client,
        bucket_name=bucket_name,
        s3_key=s3_key,
        local_file_path=str(destination_path),
    )

def _download_file_asset(download_href: str, destination_path: Path) -> None:
    """Copy one local-file asset to the requested output path."""
    parsed = urlparse(download_href)
    source_path = Path(parsed.netloc + parsed.path) if parsed.scheme.lower() == "file" else Path(download_href)
    if not source_path.is_file():
        raise FileNotFoundError(f"Source file not found: {source_path}")

    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source_path, destination_path)



In [ ]:
catalog = CatalogSearchUtils(catalog_endpoint=catalogue_endpoint, client_id=client_id, verbose=True)
start_date = "2023-11-01"
end_date = "2024-10-31"
search_body = {
    "collections": ["PL_BSM"]
}
if start_date or end_date:
    search_body["filter"] = {"op": "and", "args": []}
    if start_date:
        search_body["filter"]["args"].append({"op": ">=", "args": [{"property": "datetime"}, f"{start_date}T00:00:00Z"]})
    if end_date:
        search_body["filter"]["args"].append({"op": "<=", "args": [{"property": "datetime"}, f"{end_date}T23:59:59Z"]})

if geometry_geojson:
    if "filter" not in search_body:
        search_body["filter"] = {"op": "and", "args": []}
    search_body["filter"]["args"].append({"op": "s_intersects", "args": [{"property": "geometry"}, geometry_geojson]})

found_items = catalog.fetch_features(search_body=search_body, page_retry_attempts=5)

In [ ]:
asset_key = "quad_bandmath"
for item in found_items:
    # image_to_display = item.get("assets", {}).get("quad_bandmath").get("href")
    item_title = item.get("properties", {}).get("title", "unknown")
    download_href = get_asset_download_href(item.get("assets", {}).get(asset_key, {}))
    if download_href is None:
        continue

    destination_path = output_root / f"{item_title}_{asset_key}.tif"
    scheme = urlparse(download_href).scheme.lower()
    try:
        if scheme in {"http", "https"}:
            auth_session = get_auth_session(client_id="internal")
            _download_http_asset(download_href, destination_path, auth_session, timeout)
        elif scheme == "s3":
            s3_client = get_boto3_client()
            _download_s3_asset(download_href, destination_path, s3_client, download_s3_file)
        elif scheme in {"file", ""} or len(scheme) == 1:
            _download_file_asset(download_href, destination_path)
        else:
            print(f"Unsupported scheme {scheme} for asset {asset_key}")
            continue
        print(f"Downloaded {destination_path}")
    except Exception as exc:  # Continue with the item's remaining assets.
        print(f"Failed {asset_key}: {exc}")


In [ ]:
catalog = CatalogSearchUtils(catalog_endpoint=catalogue_endpoint, client_id=client_id, verbose=True)
start_date = "2023-11-01"
end_date = "2024-10-31"
search_body = {
    "collections": ["PL_BSM"]
}
if start_date or end_date:
    search_body["filter"] = {"op": "and", "args": []}
    if start_date:
        search_body["filter"]["args"].append({"op": ">=", "args": [{"property": "datetime"}, f"{start_date}T00:00:00Z"]})
    if end_date:
        search_body["filter"]["args"].append({"op": "<=", "args": [{"property": "datetime"}, f"{end_date}T23:59:59Z"]})

if geometry_geojson:
    if "filter" not in search_body:
        search_body["filter"] = {"op": "and", "args": []}
    search_body["filter"]["args"].append({"op": "s_intersects", "args": [{"property": "geometry"}, geometry_geojson]})

found_items = catalog.fetch_features(search_body=search_body, page_retry_attempts=5)